# Deploying MLflow models to SageMaker AI Hosting — Setup and training

This repository accompanies the blog post *"Deploying MLflow models to Amazon
SageMaker AI Hosting"*. It demonstrates the three patterns available today to move a
model from **managed MLflow on Amazon SageMaker AI** to a real-time SageMaker AI
endpoint:

| # | Pattern | Notebook |
|---|---------|----------|
| 1 | **MLflow-native deployment** (`mlflow.deployments`) | `01_deploy_mlflow_native.ipynb` |
| 2 | **Model Registry sync + `ModelBuilder` repack** | `02_deploy_modelbuilder_repack.ipynb` |
| 3 | **Model Registry sync + inference specification logging** (`sagemaker-mlflow`) | `03_deploy_inference_spec_logging.ipynb` |

This notebook is the **shared foundation**: it installs pinned dependencies, creates
(or finds) a managed MLflow app with **automatic model registration** enabled, and
trains a scikit-learn model, logging it to MLflow. Training runs inline in the
notebook kernel by default; uncommenting the `@remote` decorator in Step 2 promotes
it to a SageMaker Training Job on ephemeral managed compute.
All three patterns deploy **the same model**, so the differences you see are purely
about the deployment mechanism.

> **Important:** the model is trained **once** but logged **three times** — one
> logged MLflow model per pattern, under pattern-specific names — and it is **not
> registered** here: each pattern notebook registers *its own* logged model under a
> distinct name. Two reasons. First, the Model Registry sync (patterns 2 and 3) is
> triggered at registration time, and pattern 3 must log an inference specification
> *before* registering. Second, that specification attaches to the *logged model* —
> if the patterns shared one logged model, running pattern 3 first would make the
> spec ride along into every later registration. Separate logged models keep the
> pattern notebooks truly independent: any order, any subset.

Run this notebook in a **SageMaker Studio JupyterLab space**.

## Prerequisites

- A **SageMaker AI domain** with a user profile. The domain **execution role** needs
  permissions to manage MLflow apps, models, endpoint configs and endpoints, and
  read/write the default SageMaker bucket — plus, if you enable the `@remote`
  decorator in Step 2, permissions to run SageMaker Training Jobs
  (`sagemaker:CreateTrainingJob`, `iam:PassRole` on itself).
- Nothing to install beforehand — Step 0 installs pinned dependencies.

> **Cost note:** training runs inline by default (no extra compute); with `@remote`
> enabled it runs on an `ml.m5.xlarge` instance for a few minutes. The pattern
> notebooks each create a real-time endpoint — run `04_cleanup.ipynb` when you are
> done.

## Step 0: Install pinned dependencies

| Package            | Pin            | Why                                                                 |
|--------------------|----------------|---------------------------------------------------------------------|
| `sagemaker`        | `>=3,<4`       | SDK v3 API surface used throughout (breaking changes vs v2)          |
| `mlflow`           | `>=3.14,<4`    | MLflow 3.x client, matching the managed MLflow app                   |
| `sagemaker-mlflow` | `>=0.5.0,<1`   | `evaluate()` / `log_inference_specification()` and MLflow 3 support  |
| `scikit-learn`     | `>=1.4,<1.5`   | Matches the SKLearn serving container, so the pickled model unpickles cleanly at inference time |
| `shap`, `matplotlib` | latest       | Used by `sagemaker_mlflow.evaluate()` for the evaluation model card   |

The same `requirements.txt` is passed to the `@remote` training job when you enable
the decorator (see Step 2), so the job environment matches the notebook environment.

In [ ]:
REQUIREMENTS = """\
--extra-index-url https://download.pytorch.org/whl/cpu

sagemaker>=3,<4
mlflow>=3.14,<4
sagemaker-mlflow>=0.5.0,<1
scikit-learn>=1.4,<1.5
shap
matplotlib
"""
with open("requirements.txt", "w") as f:
    f.write(REQUIREMENTS)
print(open("requirements.txt").read())

In [ ]:
import shutil
import sys

if shutil.which("uv"):
    !uv pip install --python {sys.executable} -r requirements.txt
else:
    # Classic SageMaker notebook instances don't ship uv — fall back to pip.
    !{sys.executable} -m pip install -r requirements.txt
print("Dependencies installed. If mlflow or sagemaker was upgraded, restart the kernel before continuing.")

In [ ]:
import json
import os
import time

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sm_session = Session()
REGION = sm_session.boto_region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
try:
    EXECUTION_ROLE = get_execution_role()
except Exception:
    # Outside Studio the caller is often an IAM user, not a role. Use the
    # stack's SageMakerExecutionRoleArn output (see cfn/ README).
    EXECUTION_ROLE = os.environ['EXECUTION_ROLE']
bucket_name = sm_session.default_bucket()

print(f"Account: {ACCOUNT_ID}, Region: {REGION}")
print(f"Execution role: {EXECUTION_ROLE}")

## Step 1: Create an MLflow app with Model Registry sync enabled

Automatic model registration (`ModelRegistrationMode = AutoModelRegistrationEnabled`)
is **opt-in**. When enabled, every `mlflow.register_model()` call also creates a
Model Package Group and Model Package version in the **SageMaker AI Model Registry**
— carrying over metrics, evaluation results, lineage, and (if logged) the inference
specification.

Pattern 1 does not strictly need the sync, but creating a single app with it enabled
keeps this walkthrough simple and mirrors what you would run in a real environment.

The MLflow app's IAM **service role** must be allowed to register models and create
lineage (`sagemaker:CreateModelPackageGroup`, `sagemaker:CreateModelPackage`,
`sagemaker:UpdateModelPackage`, `sagemaker:AddTags`, lineage actions, and S3 access
to the artifact store).

In [ ]:
from sagemaker.core.resources import MlflowApp

mlflow_app_name = "deploy-mlflow-models-app"

mlflow_app = None
try:
    for app in MlflowApp.get_all():
        if app.status in ("Created", "Updated") and app.name == mlflow_app_name:
            mlflow_app = app
            print(f"Found existing MLflow app: {mlflow_app.name}")
            break
except Exception as e:
    print(f"No matching MLflow app found: {e}")

if mlflow_app is None:
    print(f"Creating MLflow app: {mlflow_app_name} (takes a few minutes)...")
    mlflow_app = MlflowApp.create(
        name=mlflow_app_name,
        artifact_store_uri=f"s3://{bucket_name}/mlflow",
        role_arn=EXECUTION_ROLE,
        model_registration_mode="AutoModelRegistrationEnabled",
    )

sm_client = boto3.client("sagemaker")
while sm_client.describe_mlflow_app(Arn=mlflow_app.arn)["Status"] not in ("Created", "Updated"):
    print("Waiting for MLflow app...")
    time.sleep(15)

MLFLOW_APP_ARN = mlflow_app.arn
print(f"MLflow app ready: {MLFLOW_APP_ARN}")

## Step 2: Train the model and log to MLflow

The training function is written to run as a SageMaker Training Job on ephemeral
managed compute: it is decorated with **`@remote`**
(`sagemaker.train.remote_function` in SDK v3). The decorator ships **commented out**
so the function runs inline in the notebook kernel — faster for iterating on this
walkthrough — and uncommenting the single `@remote` line promotes it to a real
Training Job with no other change. Either way it:

1. trains a `RandomForestRegressor` on a synthetic regression dataset,
2. logs params and **computed** train/test RMSE,
3. logs the trained model **three times — one logged model per pattern**
   (`sklearn-model-native`, `sklearn-model-modelbuilder`, `sklearn-model-infspec`) —
   with **`serialization_format="pickle"`**: this produces a plain `model.pkl`
   instead of MLflow's default `model.skops`, which the SageMaker SKLearn serving
   container cannot read (this matters for pattern 3),
4. runs `sagemaker_mlflow.evaluate()` on each logged model, so every synced Model
   Package carries an evaluation model card.

The function returns the MLflow `run_id` and one `model_id` per pattern; the pattern
notebooks pick these up via `%store`.


In [ ]:
from sagemaker.train.remote_function import remote

# Uncomment the decorator to run training as a real SageMaker Training Job on
# ephemeral managed compute (@remote cloudpickles the function and its deps).
# It is left commented here so the model trains inline in the notebook kernel,
# which is faster for iterating on this walkthrough. The function body is
# identical either way.
# @remote(instance_type="ml.m5.xlarge", dependencies="./requirements.txt")
def train_model(mlflow_app_arn, experiment_name, params):
    """Train once, then log one MLflow model per deployment pattern."""
    import mlflow
    import numpy as np
    import pandas as pd
    import sagemaker_mlflow
    from mlflow.models import infer_signature
    from sklearn.datasets import make_regression
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_squared_error
    from sklearn.model_selection import train_test_split

    mlflow.set_tracking_uri(mlflow_app_arn)
    mlflow.set_experiment(experiment_name)

    X, y = make_regression(
        n_samples=500, n_features=4, n_informative=2, noise=0.5, random_state=0
    )
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    with mlflow.start_run(run_name="candidate-model") as run:
        model = RandomForestRegressor(**params).fit(X_train, y_train)
        signature = infer_signature(X_train, model.predict(X_train))
        mlflow.log_params(params)

        train_rmse = float(np.sqrt(mean_squared_error(y_train, model.predict(X_train))))
        test_rmse = float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
        mlflow.log_metric("train_rmse", train_rmse)
        mlflow.log_metric("test_rmse", test_rmse)

        eval_df = pd.DataFrame(X_test, columns=["f1", "f2", "f3", "f4"])
        eval_df["target"] = y_test
        dataset = mlflow.data.from_pandas(eval_df, name="eval_set", targets="target")

        # One logged model per deployment pattern. Pattern 3 attaches an
        # inference specification to *its* logged model before registering;
        # separate logged models keep the pattern notebooks independent of
        # each other (any order, any subset).
        model_ids = {}
        for pattern in ("native", "modelbuilder", "infspec"):
            model_info = mlflow.sklearn.log_model(
                model,
                name=f"sklearn-model-{pattern}",
                signature=signature,
                input_example=X_train[:3],
                # Plain pickle (not skops) so the SageMaker SKLearn serving
                # container can load the artifact without extra dependencies.
                serialization_format="pickle",
            )
            # Evaluation metrics -> surfaced as a model card on the synced
            # Model Package for this pattern.
            sagemaker_mlflow.evaluate(model_info, data=dataset, model_type="regressor")
            model_ids[pattern] = model_info.model_id

        return run.info.run_id, model_ids, train_rmse, test_rmse


run_id, model_ids, train_rmse, test_rmse = train_model(
    MLFLOW_APP_ARN,
    "deploy-mlflow-models",
    {"n_estimators": 50, "random_state": 42},
)
print(f"MLflow run: {run_id}")
for pattern, mid in model_ids.items():
    print(f"Logged model ({pattern}): {mid}")
print(f"train_rmse={train_rmse:.4f}  test_rmse={test_rmse:.4f}")

After this cell, the MLflow UI shows the three logged models — one per pattern,
all from the same training run:

![Three logged models in the MLflow UI: sklearn-model-native, sklearn-model-modelbuilder, sklearn-model-infspec](img/logged-models.png)

## Navigate to MLflow

In [ ]:
from IPython.display import Javascript, HTML
import mlflow

# get the presigned url to open the MLflow UI
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=MLFLOW_APP_ARN,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

display(Javascript('window.open("{}");'.format(presigned_url)))

Access the MLflow run where we logged the metrics, the model, and the evaluation

In [ ]:
experiment_id = mlflow.get_experiment_by_name("deploy-mlflow-models").experiment_id

# get the last run in MLflow
last_run_id = mlflow.search_runs(
    experiment_ids=[experiment_id], 
    max_results=1, 
    order_by=["attributes.start_time DESC"]
)['run_id'][0]

mlflow_run_link = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/runs/{last_run_id}?workspace=default"

# first open the MLflow UI - you can close a new opened window
display(Javascript('window.open("{}");'.format(mlflow_run_link)))

## Step 3: Store shared variables for the pattern notebooks

In [ ]:
MODEL_BASE_NAME = "deploy-mlflow-demo"

mlflow_app_arn = MLFLOW_APP_ARN
mlflow_run_id = run_id
mlflow_model_ids = model_ids
model_base_name = MODEL_BASE_NAME
execution_role = EXECUTION_ROLE
region = REGION
account_id = ACCOUNT_ID

%store mlflow_app_arn
%store mlflow_app_name
%store mlflow_run_id
%store mlflow_model_ids
%store model_base_name
%store execution_role
%store region
%store account_id
%store bucket_name

## Next

Continue with any of the pattern notebooks — they are independent of each other:

- `01_deploy_mlflow_native.ipynb` — deploy straight from MLflow (the SageMaker
  Model Registry is not used)
- `02_deploy_modelbuilder_repack.ipynb` — sync to the Model Registry, repack with
  `ModelBuilder`
- `03_deploy_inference_spec_logging.ipynb` — sync to the Model Registry with a logged
  inference specification

When finished, run `04_cleanup.ipynb`.